# Glycolysis: a CRN case study

This offline example uses the reviewed KEGG M00001 fixture. It reproduces the
cached reduced-network structural results and verifies a prescribed glycolytic
flux from one glucose token. Currency removal is an explicit model choice.
These calculations do not predict kinetic or physiological behaviour.

See [the experiment commands](../../README.md) for the complete CRN studies.

Production analyses use the C++ backend through the Python interface. Execution requires a C++17 compiler and local Boost headers. It does not download dependencies.

In [1]:
from collections import Counter
from pathlib import Path
from tempfile import gettempdir
import sys

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "synkit").is_dir()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from synkit.CRN import (
    build_native, native_backend, PathwayRealizability, conserved_moieties, crnt_summary, find_siphons,
    siphon_persistence_details, crn_from_sbml, crn_to_sbml,
)
from Experiment.CRN.benchmarks import GLYCOLYSIS_FLUX, build_kegg_crn
from Experiment.CRN.studies.representation import execute

library = build_native(Path(gettempdir()) / "synkit-crn-notebook-native", with_semiflows=True)

/home/lolo/miniforge3/envs/synedu/lib/python3.11/site-packages/rxnmapper/batched_mapper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


/home/lolo/miniforge3/envs/synedu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reduced network and exact structural results

Remove the documented currency metabolites. The reduced network has 14 species,
15 directed reactions, deficiency one, three minimal conserved pools, and three
minimal siphons. Failure of the sufficient persistence test
does not establish extinction.

In [2]:
with native_backend(library):
    crn = build_kegg_crn("M00001", drop_currency=True)
    summary = crnt_summary(crn)
    moieties = conserved_moieties(crn)
    siphons = find_siphons(crn)
    persistence = siphon_persistence_details(crn)

    assert (summary.n_species, summary.n_reactions, summary.deficiency) == (14, 15, 1)
    assert not summary.is_weakly_reversible
    assert len(moieties) == len(siphons) == 3
    assert persistence.complete and persistence.persistence_ok is False
    print({
        "species": summary.n_species, "reactions": summary.n_reactions,
        "deficiency": summary.deficiency, "conserved_pools": len(moieties),
        "minimal_siphons": len(siphons), "sufficient_persistence": persistence.persistence_ok,
    })

{'species': 14, 'reactions': 15, 'deficiency': 1, 'conserved_pools': 3, 'minimal_siphons': 3, 'sufficient_persistence': False}


## Prescribed glycolytic flux

Request the prescribed reaction multiset from one alpha-D-glucose token. A positive
result supplies an ordered firing certificate. Search statistics may depend on
the implementation; the checks below concern the prescribed flux and execution.

In [3]:
with native_backend(library):
    initial_marking = {"alpha-D-Glucose": 1}
    checker = PathwayRealizability().load_syncrn_and_flow(
        crn, flow=dict(GLYCOLYSIS_FLUX), initial_marking=initial_marking,
        species="label", reaction="label",
    )
    checker.build_petri_net_from_flow()
    outcome = checker.realizability_result()
    assert outcome.status.value == "realizable"
    certificate = list(outcome.certificate)
    assert len(certificate) == sum(GLYCOLYSIS_FLUX.values()) == 15
    print({"status": outcome.status.value, "firings": len(certificate)})

{'status': 'realizable', 'firings': 15}


## Independent inventory replay

Replay the certificate against the stored reactant and product counts,
without calling the search's Petri-net firing methods. Every step must have its
reactants, all inventories must stay non-negative, and firing counts must equal
the requested flux.

In [4]:
marking = {sid: initial_marking.get(species.label, 0) for sid, species in crn.species.items()}
reactions = {reaction.label: reaction for reaction in crn.reactions.values()}
for label in certificate:
    reaction = reactions[label]
    assert all(marking[sid] >= count for sid, count in reaction.lhs.items())
    for sid, count in reaction.lhs.items():
        marking[sid] -= count
    for sid, count in reaction.rhs.items():
        marking[sid] += count
    assert all(count >= 0 for count in marking.values())
assert dict(Counter(certificate)) == dict(GLYCOLYSIS_FLUX)
final_inventory = {species.label: marking[sid] for sid, species in crn.species.items() if marking[sid]}
assert final_inventory["Pyruvate"] == 2
print(final_inventory)

{'Pyruvate': 2}


## Currency-retention control

Retain currencies under the same glucose-only inventory and prescribed flux.
The currency-retaining model has an exhaustive negative result. This comparison
shows that the reduced-model certificate does not establish execution when
cofactor requirements are restored.

In [5]:
with native_backend(library):
    intact = build_kegg_crn("M00001", drop_currency=False)
    control = execute(intact, dict(GLYCOLYSIS_FLUX), initial_marking)
    assert control["status"] == "unrealizable" and control["exhaustive"]
    print({"currencies_retained": True, "species": intact.n_species, "status": control["status"]})

{'currencies_retained': True, 'species': 24, 'status': 'unrealizable'}


## Structural SBML round trip

Serialize and reload the reduced network in memory, then compare all directed
reaction equations and structural invariants. The adapter exchanges supported
SBML Core structure. This example supplies no kinetic laws.

In [6]:
with native_backend(library):
    sbml = crn_to_sbml(crn, model_id="kegg_m00001")
    restored = crn_from_sbml(sbml)
    assert sorted(restored.to_equations(species="label", include_id=False)) == sorted(
        crn.to_equations(species="label", include_id=False)
    )
    restored_summary = crnt_summary(restored)
    assert (restored_summary.n_species, restored_summary.n_reactions, restored_summary.deficiency) == (14, 15, 1)
    print({"sbml_round_trip": "verified", "species": restored.n_species, "reactions": restored.n_reactions})

{'sbml_round_trip': 'verified', 'species': 14, 'reactions': 15}
